# Fermi surfaces from SlakoNet (tight binding)

Sample band energies on a full Brillouin-zone grid with a SlakoNet Slater-Koster
model and extract the Fermi surface. The Fermi level is set from band filling (number
of valence electrons), self-consistently with the eigenvalue spectrum. Compare the
shapes with the reference images at https://www.hzdr.de/projects/fermisur/ .

Run from the repo root (so `import fermisurf` resolves) with `slakonet`, `torch`,
`scikit-image`, `plotly`, and `jarvis-tools` installed.

In [ ]:
import sys
sys.path.insert(0, "..")  # repo root

from slakonet.optim import get_atoms
from fermisurf.slakonet_fermi import eigenfield
from fermisurf.isosurface import fermi_surface_figure

def fermi_surface(jid, nk=24, device="cpu"):
    atoms = get_atoms(jid)[0]
    field, efermi, cell = eigenfield(atoms, nk=nk, device=device)
    nb = field.shape[3]
    crossing = [b for b in range(nb) if field[..., b].min() < efermi < field[..., b].max()]
    name = atoms.composition.reduced_formula
    print(f"{name} ({jid}): E_F = {efermi:.3f} eV | bands crossing E_F: {crossing}")
    return fermi_surface_figure(field, efermi, cell, title=f"{name} ({jid}) - SlakoNet")

## Aluminum (JVASP-816)

fcc Al, 3 valence electrons: the nearly-free-electron 2nd/3rd-band sheets.

In [ ]:
fermi_surface("JVASP-816", nk=24).show()

## Other metals

Niobium (bcc) has a rich multi-sheet surface; magnesium and titanium are hexagonal.
These JIDs are the elemental ground states in JARVIS-DFT; pick a denser `nk` for a
smoother surface.

In [ ]:
metals = {
    "Al": "JVASP-816",      # fcc
    "Nb": "JVASP-14818",    # bcc
    "Mg": "JVASP-194201",    # hcp (swap for another JARVIS Mg id if needed)
    "Ti": "JVASP-194005",    # hcp
}

fig = fermi_surface(metals["Nb"], nk=24)
fig.show()

In [ ]:
# save any surface to a self-contained interactive HTML
from fermisurf.isosurface import save
save(fig, "Nb_fermi.html")
print("wrote Nb_fermi.html")

The SlakoNet surfaces reproduce the Fermi-surface topology; they are a tight-binding
surrogate, so for quantitative DFT surfaces use the companion
`jarvis_dft_fermi_surface.ipynb` (VASP + IFermi).